# Orbit fitting with `spacerocks.orbfit`

This notebook pulls astrometry from the Minor Planet Center and fits orbits with `spacerocks.orbfit`:

1. **(3666) Holman**, a main-belt asteroid: the full pipeline, outlier rejection, residuals, and a comparison with the MPC's own orbit.
2. **(99942) Apophis**: optical astrometry plus JPL radar, with the Yarkovsky parameter A2 fitted, then the 2029 close approach.
3. **(90377) Sedna**: how well a short arc pins down a distant orbit, including the Bernstein–Khushalani initial orbit.
4. **Several objects at once** with `fit_many`.

Downloads are cached in `mpc_cache/`, so rerunning the notebook doesn't hit the APIs again. Delete a file there to refresh it.

The data come from:
- the MPC [Observations API](https://docs.minorplanetcenter.net/mpc-ops-docs/apis/get-obs/) (astrometry, ADES);
- the MPC [Orbits API](https://www.minorplanetcenter.net/mpcops/documentation/orbits-api/) (the MPC's orbit, for comparison);
- JPL's [radar astrometry API](https://ssd-api.jpl.nasa.gov/doc/sb_radar.html) and [SBDB API](https://ssd-api.jpl.nasa.gov/doc/sbdb.html) (radar and JPL's Yarkovsky estimate for Apophis).

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from spacerocks import SpaceRock, orbfit
from spacerocks.spice import SpiceKernel
from spacerocks.time import Time
from spacerocks.observing import Observatory

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

kernel = SpiceKernel.defaults()   # planets, 16 massive asteroids, Earth orientation, leap seconds

## Getting the data

`get_obs` returns an object's full MPC astrometry as a DataFrame. `astrometry` turns it into what `orbfit.fit` wants:
- UTC Julian dates;
- RA/Dec in radians;
- on-sky 1σ uncertainties in radians;
- MPC station codes.

It keeps ground-based stations, and detections that come with their observer's position (ADES `sys`, `pos1`–`pos3`): satellites such as TESS and WISE, and roving observers. `orbfit.observers` turns those into barycentric observer states, as layup does, and the table carries them in its `ox`…`ovz` columns.

Stellar occultations report the occulted star's position and the object's offset from it (`raStar`, `decStar`, `deltaRA`, `deltaDec`) instead of an RA/Dec. `orbfit.occultation_radec` turns those into the object's RA/Dec, and they keep their reported uncertainties, which are a few milliarcseconds.

`debias=True` removes star-catalog biases (Eggl et al. 2020) with `orbfit.debias`. The first time, it downloads JPL's bias table, which is large.

There are two choices of weights:
- `weights="reported"` (the default): the reported `rmsRA`/`rmsDec`, or 1″ where none is reported, floored at 0.1″, because very small reported uncertainties are usually optimistic.
- `weights="veres"`: `orbfit.veres_sigma`, the Vereš et al. (2017) uncertainties by station, date, star catalog and program, exactly as layup assigns them with `weight_data=True`.

In [ ]:
CACHE = Path("mpc_cache")
CACHE.mkdir(exist_ok=True)
ARCSEC = np.pi / (180 * 3600)      # radians
AU_KM = 149597870.7


def cached(name, fetch):
    # A JSON download, cached on disk.
    path = CACHE / f"{name}.json"
    if not path.exists():
        path.write_text(json.dumps(fetch()))
    return json.loads(path.read_text())


def get_obs(desig):
    # All MPC astrometry of an object (ADES), with lower-case column names.
    def fetch():
        r = requests.get("https://data.minorplanetcenter.net/api/get-obs",
                         json={"desigs": [desig], "output_format": ["ADES_DF"]}, timeout=300)
        r.raise_for_status()
        return r.json()[0]["ADES_DF"]
    df = pd.DataFrame(cached(f"obs_{desig}", fetch))
    df.columns = df.columns.str.lower()
    return df


def get_mpc_orbit(desig):
    # The MPC's orbit (mpc_orb JSON).
    def fetch():
        r = requests.get("https://data.minorplanetcenter.net/api/get-orb", json={"desig": desig}, timeout=120)
        r.raise_for_status()
        return r.json()[0]["mpc_orb"]
    orb = cached(f"orb_{desig}", fetch)
    while isinstance(orb, list):                     # the API wraps the orbit in a list
        orb = orb[0]
    return orb


_ground = {}
def is_ground(code):
    # True for fixed ground-based stations (they have geodetic coordinates).
    if code not in _ground:
        try:
            _ground[code] = Observatory.from_obscode(code).lat is not None
        except Exception:
            _ground[code] = False
    return _ground[code]


def utc_jd(t):
    # pandas UTC timestamps -> UTC Julian dates.
    return 2451545.0 + (t - pd.Timestamp("2000-01-01T12:00:00", tz="UTC")) / pd.Timedelta(days=1)


OBS = ["ox", "oy", "oz", "ovx", "ovy", "ovz"]           # observer state columns (AU, AU/day)


def astrometry(df, start=None, end=None, weights="reported", debias=False, sigma_default=1.0, sigma_floor=0.1):
    # MPC ADES rows -> a table of optical astrometry with observer states, sorted by time.
    # Keeps ground stations, and detections whose observer position comes with them (ADES
    # sys/pos1..3: satellites such as TESS and WISE, roving observers).
    # weights: "reported" (rmsRA/rmsDec, sigma_default where missing, floored at sigma_floor)
    # or "veres" (Vereš et al. 2017 by station, date, catalog and program, as layup assigns them).
    # debias: remove star-catalog biases (Eggl et al. 2020); downloads JPL's table the first time.
    col = lambda c: pd.to_numeric(df[c], errors="coerce") if c in df else pd.Series(np.nan, index=df.index)
    t = pd.to_datetime(df["obstime"], format="mixed", utc=True)
    ra, dec = col("ra"), col("dec")
    stn = df["stn"].astype(str)
    pos = np.column_stack([col(c) for c in ("pos1", "pos2", "pos3")])
    has_pos = np.isfinite(pos).all(axis=1) & (df["sys"].isin(["ICRF_KM", "ICRF_AU", "WGS84"]) if "sys" in df else False)
    # Occultations: the object's position is the occulted star's plus the reported offset.
    occ = ra.isna() & col("rastar").notna() & col("decstar").notna()
    if occ.any():
        r, d = orbfit.occultation_radec(np.radians(col("rastar")[occ]), np.radians(col("decstar")[occ]),
                                        np.radians(col("deltara")[occ].fillna(0) / 3600),
                                        np.radians(col("deltadec")[occ].fillna(0) / 3600))
        ra[occ], dec[occ] = np.degrees(r), np.degrees(d)
    keep = ra.notna() & dec.notna() & (has_pos | stn.map(is_ground))
    if "deprecated" in df:                            # detections the MPC has withdrawn
        keep &= df["deprecated"].isna()
    if start is not None:
        keep &= t >= pd.Timestamp(start, tz="UTC")
    if end is not None:
        keep &= t < pd.Timestamp(end, tz="UTC")
    sra = col("rmsra").fillna(sigma_default).clip(lower=sigma_floor)
    sdec = col("rmsdec").fillna(sigma_default).clip(lower=sigma_floor)
    sra, sdec = sra * ARCSEC, sdec * ARCSEC
    if weights == "veres":
        opt = lambda c: list(df[c].where(df[c].notna(), None)) if c in df else None
        sra = sdec = pd.Series(orbfit.veres_sigma(list(stn), utc_jd(t).values, catalog=opt("astcat"), program=opt("prog")), index=df.index)
    # Occultations keep their reported (milliarcsecond) uncertainties, without the floor.
    sra = sra.where(~occ, col("rmsra") * ARCSEC)
    sdec = sdec.where(~occ, col("rmsdec") * ARCSEC)
    ra, dec = ra * np.pi / 180, dec * np.pi / 180
    if debias and "astcat" in df:
        cat = list(df["astcat"].where(df["astcat"].notna() & keep, None))       # rows we keep only
        ra_d, dec_d = orbfit.debias(ra.fillna(0).values, dec.fillna(0).values, utc_jd(t).values, cat)
        ra, dec = ra.where(~keep, ra_d), dec.where(~keep, dec_d)
    out = pd.DataFrame({"time": t, "jd": utc_jd(t), "ra": ra, "dec": dec,
                        "sigma_ra": sra, "sigma_dec": sdec, "stn": stn})[keep]
    k = keep.values
    sys = [s if p else None for s, p in zip(df["sys"] if "sys" in df else [None] * len(df), has_pos)]
    vel = np.column_stack([col(c) for c in ("vel1", "vel2", "vel3")])
    out[OBS] = orbfit.observers(list(out["stn"]), out["jd"].values, kernel, sys=[s for s, kk in zip(sys, k) if kk],
                                pos=pos[k], vel=vel[k])
    print(f"{keep.sum()} of {len(df)} detections kept ({int((has_pos & keep).sum())} with their own observer position, "
          f"{int((occ & keep).sum())} occultations)")
    return out.sort_values("jd").reset_index(drop=True)

### Fitting helpers

`fit_table` passes a table like the one above to `orbfit.fit`. If the table has radar columns, those go in too.

On real MPC data we fit with `robust=True`. On its own, `orbfit.fit` reproduces layup's pipeline exactly, and on raw survey data that pipeline is fragile:
- It does no outlier rejection. An initial orbit is accepted only if its fit to the longest arc has χ²/ndof ≤ 10, so one bad detection in that arc can sink the whole fit (flag 3).
- It fits each Gauss starting orbit to the whole longest arc at once. For a near-Earth object observed for months, that fit diverges even from a good starting orbit. Bennu's 313-day 2011–12 arc is an example.

`robust=True` fixes both:
- Detections more than `outlier_sigma` (default 4σ) from the orbit are left out, and all of them are re-evaluated every round, so rejected ones can come back.
- If layup's pipeline gives no orbit, the fit starts from a window of at most 60 days (the one with the most detections that works) and widens it step by step.

`fit.used` marks the detections in the final fit, and `fit.residuals` covers all of them, the rejected ones included.

In [ ]:
def radar_kwargs(d):
    if "delay" not in d:
        return {}
    return dict(delay=d["delay"].values, sigma_delay=d["sigma_delay"].values,
                doppler=d["doppler"].values, sigma_doppler=d["sigma_doppler"].values,
                frequency=d["freq"].values, transmitter=list(d["trx"]))


def fit_table(d, **kw):
    return orbfit.fit(d["ra"].values, d["dec"].values, d["jd"].values, d[OBS].values, kernel,
                      sigma_ra=d["sigma_ra"].values, sigma_dec=d["sigma_dec"].values,
                      **radar_kwargs(d), **kw)


def fit_robust(d, **kw):
    # A robust fit, with a one-line report.
    t0 = time.time()
    fit = fit_table(d, robust=True, **kw)
    used = fit.used.sum() if len(fit.used) else 0
    print(f"{fit}\n{used} of {len(d)} detections used, {time.time() - t0:.1f} s")
    return fit


def summarize(fit):
    # Heliocentric ecliptic elements of a fit.
    rock = fit.rock
    rock.to_helio(kernel)
    rock.change_reference_plane("ECLIPJ2000")
    print(f"epoch {fit.epoch.utc().iso()} UTC   chi2/ndof = {fit.chi2:.1f}/{fit.ndof} = {fit.chi2 / fit.ndof:.2f}")
    print(f"a = {rock.a():.8f} AU   e = {rock.e():.8f}   i = {np.degrees(rock.inc()):.6f} deg")

## 1. A main-belt asteroid: (3666) Holman

The full pipeline runs in several steps:
1. Split the detections into arcs.
2. Try Gauss's method on triplets from the longest arc, screening the candidates with short fits (and falling back to the Bernstein–Khushalani orbit if none works).
3. Fit the winner to everything.

We use data from 2000 on. Older photographic astrometry would need proper weights.

In [ ]:
holman = astrometry(get_obs("3666"), start="2000-01-01")
holman.groupby(holman["time"].dt.year).size().plot.bar(figsize=(10, 2.5), ylabel="detections");

In [ ]:
fit = fit_robust(holman)
summarize(fit)

`fit.residuals` has one row per detection. The columns are:
- RA·cos Dec and Dec, in radians;
- RA and Dec rates, in radians/day;
- radar delay, in seconds, and Doppler, in Hz.

Columns a detection didn't measure are NaN. With `robust=True`, the rows are at the final orbit and include the rejected detections.

In [ ]:
res, keep = fit.residuals / ARCSEC, fit.used

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, k, label in zip(axes, (0, 1), ("RA cos Dec", "Dec")):
    y = np.clip(res[:, k], -2.9, 2.9)                  # off-scale points drawn at the edge
    ax.scatter(holman["time"][keep], y[keep], s=2, label="used")
    ax.scatter(holman["time"][~keep], y[~keep], s=6, c="r", label="rejected")
    ax.set_ylabel(f"{label} O-C (″)")
    ax.set_ylim(-3, 3)
axes[0].legend(loc="upper left", markerscale=3)
print("rms of used residuals: RA %.3f″, Dec %.3f″" % tuple(np.sqrt(np.mean(res[keep, :2]**2, axis=0))))

### Keeping the orbit up to date

When new detections come in, the fit need not start over. Pass the earlier fit as `prior`. The new detections are then fitted alone, with the old fit's covariance as a Gaussian prior (layup's sequential update). `fit.route` says what happened: `"sequential"` here. The result matches a full refit to a small fraction of σ, while integrating only the new detections.

In [ ]:
cut = holman["time"] < "2023-01-01"
t0 = time.time(); old = fit_table(holman[cut]); t_old = time.time() - t0
t0 = time.time(); upd = fit_table(holman, prior=old); t_upd = time.time() - t0
full = fit_table(holman, initial=old.rock)
d = upd.state - full.state
print(f"{cut.sum()} old + {(~cut).sum()} new detections: route {upd.route!r}, {t_upd:.2f} s (the old fit took {t_old:.2f} s)")
print(f"update vs full refit: {np.sqrt(d @ np.linalg.solve(full.state_covariance, d)):.1e} sigma")

### Where to look, and how well we know it

`orbfit.predict` gives the astrometric position of the fitted orbit for an observer at any time, and maps the fit's covariance onto the sky: a 1σ error ellipse (`sigma_major`, `sigma_minor` in arcsec, `pa` in degrees). This is layup's `predict`.

In [ ]:
when = [Time.from_isot(f"{y}-01-01T06:00:00") for y in range(2026, 2036, 2)]
p = orbfit.predict(fit, when, "X05", kernel)
pd.DataFrame({"date": [t.utc().iso()[:10] for t in when], "RA [deg]": np.degrees(p["ra"]),
              "Dec [deg]": np.degrees(p["dec"]), "Delta [AU]": p["delta"],
              "sigma major [\"]": p["sigma_major"], "sigma minor [\"]": p["sigma_minor"], "PA [deg]": p["pa"]})

### Comparison with the MPC's orbit

The MPC gives a heliocentric ecliptic Cartesian state at an epoch in TT. We convert it to barycentric J2000 and propagate it to our epoch with ASSIST. Then we measure the difference in kilometres and in σ of our covariance, via the Mahalanobis distance over 6 dof.

The two fits use different weights, outlier rejection and data, so differences of a few σ are expected.

In [ ]:
def mpc_rock(orb):
    # The MPC's orbit as a barycentric J2000 SpaceRock.
    x = orb["CAR"]["coefficient_values"][:6]
    ep = orb["epoch_data"]
    scale = {"TDT": "tt", "TT": "tt", "TDB": "tdb", "UTC": "utc"}[ep.get("timesystem", "TT").upper()]
    rock = SpaceRock.from_xyz("mpc", *x, Time(ep["epoch"], scale, "mjd"), "ECLIPJ2000", "SUN")
    rock.to_ssb(kernel)
    rock.change_reference_plane("J2000")
    return rock


def compare(fit, other):
    # Difference between a fit and another SpaceRock (propagated to the fit's epoch).
    other.propagate(fit.epoch, kernel)
    d = np.array([*other.position, *other.velocity]) - fit.state
    sig = np.sqrt(d @ np.linalg.solve(fit.state_covariance, d))
    s = np.sqrt(np.diag(fit.state_covariance))
    print(f"position difference {np.linalg.norm(d[:3]) * AU_KM:.2f} km "
          f"(our 1-sigma: {np.linalg.norm(s[:3]) * AU_KM:.2f} km)")
    print(f"velocity difference {np.linalg.norm(d[3:]) * AU_KM / 86400 * 1e5:.3f} cm/s")
    print(f"Mahalanobis distance {sig:.2f} sigma (6 dof)")


orb = get_mpc_orbit("3666")
print(orb["epoch_data"], orb["system_data"], sep="\n")
compare(fit, mpc_rock(orb))

The same fit with Vereš et al. (2017) weights instead of the reported uncertainties:

In [ ]:
holman_veres = astrometry(get_obs("3666"), start="2000-01-01", weights="veres")
print("sigma (arcsec): median %.2f, range %.2f-%.2f" % tuple(np.array([np.median(holman_veres["sigma_ra"]),
      holman_veres["sigma_ra"].min(), holman_veres["sigma_ra"].max()]) / ARCSEC))
fit_v = fit_robust(holman_veres)
summarize(fit_v)
compare(fit_v, mpc_rock(orb))

And with the star-catalog biases removed too. This is off by default, because it downloads JPL's bias table (several hundred MB unpacked); set `DEBIAS = True` to run it.

In [ ]:
DEBIAS = False
if DEBIAS:
    holman_db = astrometry(get_obs("3666"), start="2000-01-01", weights="veres", debias=True)
    moved = np.hypot((holman_db["ra"] - holman_veres["ra"]) * np.cos(holman_veres["dec"]),
                     holman_db["dec"] - holman_veres["dec"]) / ARCSEC
    print(f"debiasing moved {np.sum(moved > 0)} detections, by up to {moved.max():.2f} arcsec")
    fit_db = fit_robust(holman_db)
    summarize(fit_db)
    compare(fit_db, mpc_rock(orb))

## 2. Radar and Yarkovsky: (99942) Apophis

JPL's radar astrometry comes as delays (µs) and Doppler shifts (Hz), with JPL station codes. `radar_table` does four things:
1. Maps JPL's station codes to MPC codes.
2. Keeps center-of-mass measurements. Peak-power ones need a shape model.
3. Converts to the units `orbfit.fit` takes: delay in seconds, frequency in Hz.
4. Puts delays and Dopplers in the same table as the optical data, with NaN in the columns a row doesn't measure.

The epoch of a radar measurement is the receive time. Its transmitter is given by code, and spacerocks computes that station's exact state at the transmit time. The optical rows get the geocenter (500) as a placeholder transmitter, which is never used.

In [ ]:
JPL_TO_MPC = {-1: "251", -2: "254", -9: "256", -13: "252", -14: "253", -25: "257", -38: "255", -73: "259"}


def get_radar(desig):
    # JPL radar astrometry as a DataFrame (empty if there is none).
    def fetch():
        r = requests.get("https://ssd-api.jpl.nasa.gov/sb_radar.api", params={"des": desig}, timeout=120)
        r.raise_for_status()
        return r.json()
    js = cached(f"radar_{desig}", fetch)
    return pd.DataFrame(js.get("data", []), columns=js.get("fields"))


def radar_table(rad):
    rad = rad[rad["bp"] == "C"].copy()
    rcvr, xmit = rad["rcvr"].astype(int).map(JPL_TO_MPC), rad["xmit"].astype(int).map(JPL_TO_MPC)
    ok = rcvr.notna() & xmit.notna()
    if (~ok).any():
        print(f"dropping {(~ok).sum()} measurements from stations without an MPC code")
    rad, rcvr, xmit = rad[ok], rcvr[ok], xmit[ok]
    t = pd.to_datetime(rad["epoch"], utc=True).reset_index(drop=True)
    value, sigma = rad["value"].astype(float).values, rad["sigma"].astype(float).values
    is_delay = (rad["units"] == "us").values
    nan = np.full(len(rad), np.nan)
    out = pd.DataFrame({
        "time": t, "jd": utc_jd(t), "ra": nan, "dec": nan, "sigma_ra": nan, "sigma_dec": nan,
        "stn": rcvr.values, "trx": xmit.values, "freq": rad["freq"].astype(float).values * 1e6,
        "delay": np.where(is_delay, value * 1e-6, np.nan), "sigma_delay": np.where(is_delay, sigma * 1e-6, np.nan),
        "doppler": np.where(is_delay, np.nan, value), "sigma_doppler": np.where(is_delay, np.nan, sigma),
    })
    out[OBS] = orbfit.observers(list(out["stn"]), out["jd"].values, kernel)
    return out


radar = radar_table(get_radar("99942"))
optical = astrometry(get_obs("99942"))
optical = optical.assign(trx="500", freq=np.nan, delay=np.nan, sigma_delay=np.nan,
                         doppler=np.nan, sigma_doppler=np.nan)
apophis = pd.concat([optical, radar]).sort_values("jd").reset_index(drop=True)
print(f"{len(optical)} optical detections, {radar['delay'].notna().sum()} delays, "
      f"{radar['doppler'].notna().sum()} Dopplers from", sorted(set(radar["stn"]) | set(radar["trx"])))

The Yarkovsky effect is modelled as a transverse acceleration A2·g(r). JPL's solution also estimates a radial term A1, which absorbs solar radiation pressure, so we fit the same model: `nongrav="A1A2"`. The pipeline first fits a gravity-only orbit, then fits A1 and A2 jointly with the state. `fit.nongrav` and `fit.nongrav_sigma` hold (A1, A2, A3) in AU/day².

Without A2, the most precise radar delays look like outliers to the gravity-only orbit. So in robust mode the joint fit starts from every detection, not just the survivors of the gravity-only fit.

A2 is measured at 190σ and agrees with JPL's to a fraction of a sigma. A1 is not detected, in either solution: both are about 1σ from zero. The two values then depend on the weights and data selection, so they differ by a fraction of their uncertainty, not to the digits A2 does. JPL debiases the astrometry and assigns its own weights. With `astrometry(..., weights="veres")` (and `debias=True` where JPL's table can be downloaded), the weighting is closer to JPL's.

In [ ]:
afit = fit_robust(apophis, nongrav="A1A2")
summarize(afit)

# JPL's solution, for comparison
sbdb = cached("sbdb_99942", lambda: requests.get("https://ssd-api.jpl.nasa.gov/sbdb.api",
                                                  params={"sstr": "99942"}, timeout=120).json())
jpl = {p["name"]: (float(p["value"]), float(p["sigma"])) for p in sbdb.get("orbit", {}).get("model_pars", [])
       if p.get("kind") == "EST"}
for k, name in enumerate(("A1", "A2")):
    ours, sig = afit.nongrav[k], afit.nongrav_sigma[k]
    line = f"{name} = {ours:+.3e} +/- {sig:.1e} AU/day^2 ({abs(ours) / sig:.1f} sigma from zero)"
    if name in jpl:
        v, s = jpl[name]
        line += f";  JPL {v:+.3e} +/- {s:.1e};  difference {abs(ours - v) / np.hypot(sig, s):.1f} sigma"
    print(line)

`nongrav="auto"` chooses the model itself, as layup's `fit_nongrav="auto"` does. It keeps the gravity-only orbit if that fit's χ²/ndof is at most 1.5. Otherwise it adopts the simplest warranted model from A2, A1, A3, then A1+A2, then all three.

That gate assumes calibrated uncertainties. The MPC's reported ones are conservative, so the gravity-only fit already has χ²/ndof ≈ 0.3, and with the default gate the A2 detection is missed. Turning the gate off (`nongrav_thresholds=(0.0, 9.0, 3.0)`) finds it, and only it.

In [ ]:
for thresholds in (None, (0.0, 9.0, 3.0)):
    f = fit_table(apophis, nongrav="auto", nongrav_thresholds=thresholds, robust=True)
    chosen = [n for n, s in zip(("A1", "A2", "A3"), f.nongrav_sigma) if np.isfinite(s)] or ["none"]
    print(f"thresholds {thresholds}: chose {'+'.join(chosen)}; chi2/ndof = {f.chi2 / f.ndof:.3f}; "
          f"A = {f.nongrav}")

In [ ]:
res, akeep = afit.residuals, afit.used
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
opt = apophis["delay"].isna() & apophis["doppler"].isna()
axes[0].scatter(res[opt & akeep, 0] / ARCSEC, res[opt & akeep, 1] / ARCSEC, s=1, alpha=0.3)
axes[0].set(xlim=(-3, 3), ylim=(-3, 3), xlabel="RA cos Dec O-C (″)", ylabel="Dec O-C (″)", title="optical")
for ax, k, col, scale, unit in ((axes[1], 4, "delay", 1e6, "µs"), (axes[2], 5, "doppler", 1, "Hz")):
    m = apophis[col].notna().values
    ax.errorbar(apophis["time"][m], res[m, k] * scale, yerr=apophis["sigma_" + col][m] * scale, fmt="o", ms=3)
    ax.axhline(0, c="k", lw=0.5)
    ax.set(ylabel=f"O-C ({unit})", title=col)
    ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()

### The 2029 close approach

We propagate the fitted orbit, with A2, to April 2029 and find the minimum geocentric distance. JPL puts the approach at about 38,000 km from Earth's center on 2029 April 13, near 21:46 UTC.

In [ ]:
rock = afit.rock
earth_dist = lambda r, t: np.linalg.norm(np.subtract(r.position, SpaceRock.from_spice("earth", t, "J2000", "SSB", kernel).position))

t = Time.from_isot("2029-04-13T12:00:00").tdb()
best = (np.inf, None)
for step in range(24 * 60 // 2):                     # every 2 minutes for a day
    rock.propagate(t, kernel)
    d = earth_dist(rock, t)
    if d < best[0]:
        best = (d, t.utc().iso())
    t = t + 2 / 1440
print(f"closest approach {best[0] * AU_KM:,.0f} km from Earth's center at {best[1]} UTC")

## 3. A short arc of a distant object: (90377) Sedna

Over a few nights, a distant object hardly curves. Its distance is poorly constrained, and Gauss's method often finds no root. The Bernstein–Khushalani IOD handles this case.

`orbfit.bk_iod` gives a linear initial orbit with zero radial velocity. `orbfit.fit` uses it automatically when every Gauss candidate fails.

We take the densest stretch of one recent opposition and fit ever-longer pieces of it. Then we compare each fit with the fit to all the data.

In [ ]:
sedna = astrometry(get_obs("90377"), start="2005-01-01")
sfit = fit_robust(sedna)
summarize(sfit)

# the 60-day window, in the last five years, with the most detections
jd = sedna["jd"].values
recent = np.where(jd > jd.max() - 5 * 365.25)[0]
counts = np.searchsorted(jd, jd[recent] + 60) - recent
t0 = jd[recent[np.argmax(counts)]]
print(f"window starts {sedna['time'][recent[np.argmax(counts)]].date()}: {counts.max()} detections in 60 days")

In [ ]:
def distance_and_sigma(state, cov):
    # Barycentric distance and its 1-sigma from the covariance.
    r = np.linalg.norm(state[:3])
    u = np.asarray(state[:3]) / r
    return r, np.sqrt(u @ cov[:3, :3] @ u)


ref = sfit.rock
rows = []
for span in (3, 7, 15, 30, 60):
    arc = sedna[(jd >= t0) & (jd <= t0 + span)]
    seed = orbfit.bk_iod(arc["ra"].values, arc["dec"].values, arc["jd"].values, arc[OBS].values, kernel,
                         sigma_ra=arc["sigma_ra"].values, sigma_dec=arc["sigma_dec"].values)
    row = {"span [d]": span, "n": len(arc), "nights": arc["time"].dt.date.nunique(),
           "BK r": np.linalg.norm(seed.position) if seed is not None else np.nan}
    fits = {"pipeline": fit_table(arc), "from BK": fit_table(arc, initial=seed) if seed is not None else None}
    for label, f in fits.items():
        row[f"{label} flag"] = f.flag if f is not None else None
        if f is None or f.flag not in (0, 2):
            continue
        r, sr = distance_and_sigma(f.state, f.state_covariance)
        ref.propagate(f.epoch, kernel)
        err = (r - np.linalg.norm(ref.position)) / sr
        row.update({f"{label} r": r, f"{label} sigma_r": sr, f"{label} error [sigma]": err})
    ref.propagate(Time(arc["jd"].values.mean(), "utc", "jd"), kernel)
    row["full-data r"] = np.linalg.norm(ref.position)
    rows.append(row)
cols = ["span [d]", "n", "nights", "BK r", "full-data r"] + [f"{p} {c}" for p in ("pipeline", "from BK")
                                                           for c in ("flag", "r", "sigma_r", "error [sigma]")]
pd.DataFrame(rows).reindex(columns=cols).round(3)

The columns are:
- `BK r`: the Bernstein–Khushalani distance, in AU.
- `pipeline flag`, `from BK flag`: the fit flags (see the table at the end).
- `pipeline r` ± `pipeline sigma_r`: the distance from `orbfit.fit` on the short arc alone.
- `from BK r` ± `from BK sigma_r`: the same fit, started from the Bernstein–Khushalani orbit.
- `error [sigma]`: how far each short-arc distance is from the full-data orbit's, in units of its own σ.
- `full-data r`: the distance from the fit to all the data.

A few nights of an object at 80+ AU don't constrain all six parameters. The full fit may then fail (e.g. flag 9, an implausible orbit), or Gauss's method may land on a spurious nearby orbit that fits the few nights equally well, which the pipeline accepts because it tries Gauss first. The Bernstein–Khushalani distance is still informative, and once an arc spans a few weeks both fits settle on the right orbit.

## 4. Many objects at once: `fit_many`

`fit_many` takes one long table with an id per detection and fits the objects in parallel. It returns plain arrays: one row per object, plus `used`, one bool per detection.

We run it twice: first as layup does, then with `robust=True`. Layup's pipeline typically loses a few objects on raw MPC data (flag 3). Here that's Bennu, whose densest arc, 2011–12, runs 313 days.

In [ ]:
names = {"3666": "Holman", "433": "Eros", "101955": "Bennu", "90377": "Sedna", "99942": "Apophis"}
tables = [astrometry(get_obs(d), start="2010-01-01").assign(id=name) for d, name in names.items()]
data = pd.concat(tables, ignore_index=True)
args = (list(data["id"]), data["ra"].values, data["dec"].values, data["jd"].values, data[OBS].values, kernel)
weights = dict(sigma_ra=data["sigma_ra"].values, sigma_dec=data["sigma_dec"].values)

t0 = time.time()
plain = orbfit.fit_many(*args, **weights)
print(f"layup's pipeline: {len(data)} detections of {len(names)} objects in {time.time() - t0:.1f} s")
t0 = time.time()
res = orbfit.fit_many(*args, **weights, robust=True)
print(f"robust:           {len(data)} detections of {len(names)} objects in {time.time() - t0:.1f} s")


def elements(state, epoch, name):
    rock = SpaceRock.from_xyz(name, *state, Time(epoch, "tdb", "jd"), "J2000", "SSB")
    rock.to_helio(kernel)
    rock.change_reference_plane("ECLIPJ2000")
    return rock.a(), rock.e(), np.degrees(rock.inc())


ok = np.isin(res["flag"], (0, 2))                   # other flags carry no usable orbit
a, e, inc = np.array([elements(s, ep, n) if good else (np.nan,) * 3
                      for s, ep, n, good in zip(res["state"], res["epoch"], res["id"], ok)]).T
used = data.assign(used=res["used"]).groupby("id", sort=False)["used"].sum().values
pd.DataFrame({"id": res["id"], "layup flag": plain["flag"], "robust flag": res["flag"],
              "n_det": data.groupby("id", sort=False).size().values, "used": used,
              "chi2/ndof": res["chi2"] / res["ndof"], "a [AU]": a, "e": e, "i [deg]": inc,
              "sigma_x [km]": np.where(ok, np.sqrt(res["covariance"][:, 0, 0]) * AU_KM, np.nan)})

### Flags

| flag | meaning |
|---|---|
| 0 | converged |
| 1 | not converged within `max_iter` |
| 2 | converged, but χ²/ndof > `chi2_threshold` (10) |
| 3 | no IOD candidate converged |
| 4 | the arc-by-arc build-up failed |
| 5 | no solution (e.g. fewer than three detections, or no IOD root) |
| 6 | degenerate covariance |
| 7, 8 | sequential update: prior covariance not positive definite / update too large (only without a refit) |
| 9 | implausible orbit |

See `docs/python-docs/orbfit.md` for the full API.